In [ ]:
!pip install jax "jax[cuda13]" optax einops transformers huggingface_hub datasets pynvml

### Gemma 3

This notebook walks you through implementing the core of the Gemma 3 transformer model from scratch using JAX. 

Gemma 3 is a family of open-weights, lightweight, and multimodal AI models from Google. Gemma 3 handles both text and image inputs to generate text, allowing for sophisticated visual understanding. Below you will find code for the Gemma 3 Text Model only. 

Gemma 3 Technical Report: https://arxiv.org/pdf/2503.19786.   
HF Source: https://huggingface.co/google/gemma-3-4B-it

### Imports

In [1]:
import os
os.environ['TF_GPU_ALLOCATOR'] = 'cuda_malloc_async'
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'

In [2]:
import glob
import json
import os
from dataclasses import dataclass
from functools import partial
from typing import Optional

import numpy as np
from PIL import Image
import jax
import jax.numpy as jnp
from jax.sharding import Mesh, NamedSharding, PartitionSpec as P
import optax
from safetensors.numpy import load_file
from transformers import AutoTokenizer
from einops import rearrange, repeat
import pynvml

/usr/local/lib/python3.11/dist-packages/torch/cuda/__init__.py:58: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [3]:
jax.devices()

[CudaDevice(id=0), CudaDevice(id=1)]

In [ ]:
import os
os.environ["HF_TOKEN"] = "hf_xxxxxxxxxxxxxxxx" # Your HF Token here

In [5]:
HF_REPO_ID = "google/gemma-3-4b-it"
LOCAL_DIR_PATH = "gemma-3-4b-it"

The below code defines the configuration for the Gemma 3 4B IT model. This allows us to easily manage and access the model's hyperparameters throughout our implementation. We use the 4B parameter configuration as a base, so that you can run the model on minimal hardware. You can adjust these parameters to match the larger configurations (3B, 7B, 30B) when needed.

In [6]:
@dataclass(frozen=True)
class Gemma3Config:
    bos_token_id: int = 2
    eos_token_id: tuple = (1, 106)
    hidden_size: int = 2560
    num_layers: int = 34
    num_heads: int = 8
    num_kv_heads: int = 4
    head_dim: int = 256
    intermediate_size: int = 10240
    max_seq_len: int = 512        # 128k context
    rms_norm_eps: float = 1e-6
    query_pre_attn_scalar: int = 256
    sliding_window: int = 1024       # larger window than 270M
    sliding_window_pattern: int = 6
    vocab_size: int = 262144
    full_attention_layers: tuple = (5, 11, 17, 23, 29, 33)  # every 6th layer in 34 layers

config = Gemma3Config()

In this section, we download the weights of the model of interest from huggingface to use in our implementation.

In [7]:
from huggingface_hub import snapshot_download

local_dir = snapshot_download(
    repo_id=HF_REPO_ID,
    local_dir=LOCAL_DIR_PATH,
)
print(f"Downloaded repository path: {local_dir}")

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

Downloaded repository path: /workspace/gemma-3-4b-it


In [8]:
tokenizer = AutoTokenizer.from_pretrained(LOCAL_DIR_PATH)
print("Tokenizer loaded successfully.")

Tokenizer loaded successfully.


### Weights Load

In [9]:
shard_paths = sorted(glob.glob(os.path.join(LOCAL_DIR_PATH, "*.safetensors")))

if not shard_paths:
    raise FileNotFoundError(f"No safetensors found in {LOCAL_DIR_PATH}")

combined_weights = {}
for path in shard_paths:
    print(f"Loading shard: {os.path.basename(path)}...")
    shard = load_file(path)
    combined_weights.update(shard)
        
hf_weights = combined_weights

Loading shard: model-00001-of-00002.safetensors...
Loading shard: model-00002-of-00002.safetensors...


In [10]:
NUM_GPU = 2

devices = np.array(jax.devices()).reshape(NUM_GPU,)
mesh = Mesh(devices, axis_names=('fsdp',))

embed = jax.P(None, "fsdp")
att_qkv = jax.P(None, None, "fsdp")
att_out = jax.P(None, "fsdp", None)
mlp_in = jax.P(None, None, "fsdp")
mlp_out = jax.P(None, "fsdp", None)
out_kernel = jax.P("fsdp", None)

act_ids = jax.P("fsdp")
act_seq = jax.P("fsdp", None, None)
act_att = jax.P("fsdp", None, None, None)
act_hidden = jax.P("fsdp", None, None)

replicate = jax.P()

In [11]:
def get_w(
    name: str,
    transpose: bool = False,
    sharding = None,
):
    
    if isinstance(name, list):
        arrays = [hf_weights.pop(n) for n in name]
        arrays = [a.T if transpose else a for a in arrays]
        val = np.stack(arrays)
    else:
        val = hf_weights.pop(name)
        val = val.T if transpose else val

    if sharding is None:
        return jnp.array(val, dtype=jnp.bfloat16)

    return jax.device_put(val, NamedSharding(mesh, sharding))

n = config.num_layers

prefix = 'language_model.model.'
layer_prefix = prefix + 'layers.'

m = {
    'embed': get_w(prefix + 'embed_tokens.weight', sharding=embed),
    'final_norm': get_w(prefix + 'norm.weight', sharding=replicate),
    'layers': {
        'attn': {
            'q_proj': get_w([f'{layer_prefix}{i}.self_attn.q_proj.weight' for i in range(n)], True, att_qkv),
            'k_proj': get_w([f'{layer_prefix}{i}.self_attn.k_proj.weight' for i in range(n)], True, att_qkv),
            'v_proj': get_w([f'{layer_prefix}{i}.self_attn.v_proj.weight' for i in range(n)], True, att_qkv),
            'o_proj': get_w([f'{layer_prefix}{i}.self_attn.o_proj.weight' for i in range(n)], True, att_out),
            'q_norm': {'scale': get_w([f'{layer_prefix}{i}.self_attn.q_norm.weight' for i in range(n)], sharding=replicate)},
            'k_norm': {'scale': get_w([f'{layer_prefix}{i}.self_attn.k_norm.weight' for i in range(n)], sharding=replicate)},
        },
        'mlp': {
            'gate_proj': get_w([f'{layer_prefix}{i}.mlp.gate_proj.weight' for i in range(n)], True, mlp_in),
            'up_proj': get_w([f'{layer_prefix}{i}.mlp.up_proj.weight'   for i in range(n)], True, mlp_in),
            'down_proj': get_w([f'{layer_prefix}{i}.mlp.down_proj.weight' for i in range(n)], True, mlp_out),
        },
        'input_norm': {'scale': get_w([f'{layer_prefix}{i}.input_layernorm.weight' for i in range(n)], sharding=replicate)},
        'post_attn_norm': {'scale': get_w([f'{layer_prefix}{i}.post_attention_layernorm.weight' for i in range(n)], sharding=replicate)},
        'pre_ff_norm': {'scale': get_w([f'{layer_prefix}{i}.pre_feedforward_layernorm.weight' for i in range(n)], sharding=replicate)},
        'post_ff_norm': {'scale': get_w([f'{layer_prefix}{i}.post_feedforward_layernorm.weight' for i in range(n)], sharding=replicate)},
    }
}

print("Successfully Loaded Weights")

Successfully Loaded Weights


In [12]:
if hf_weights.keys():
    for key in hf_weights.keys():
        if "language_model" in key:
            print(key)
else:
    print(hf_weights.keys())

In [13]:
print("Embed:", m['embed'].shape)
print("Final Norm:", m['final_norm'].shape)
print("Q_proj:", m['layers']['attn']['q_proj'].shape)
print("K_proj:", m['layers']['attn']['k_proj'].shape)
print("O_proj:", m['layers']['attn']['o_proj'].shape)
print("Q_norm scale:", m['layers']['attn']['q_norm']['scale'].shape)
print("K_norm scale:", m['layers']['attn']['k_norm']['scale'].shape)
print("Gate_proj:", m['layers']['mlp']['gate_proj'].shape)
print("Up_proj:", m['layers']['mlp']['up_proj'].shape)
print("Down_proj:", m['layers']['mlp']['down_proj'].shape)
print("Input_norm scale:", m['layers']['input_norm']['scale'].shape)
print("Post_attn_norm scale:", m['layers']['post_attn_norm']['scale'].shape)
print("Pre_ff_norm scale:", m['layers']['pre_ff_norm']['scale'].shape)
print("Post_ff_norm scale:", m['layers']['post_ff_norm']['scale'].shape)

Embed: (262208, 2560)
Final Norm: (2560,)
Q_proj: (34, 2560, 2048)
K_proj: (34, 2560, 1024)
O_proj: (34, 2048, 2560)
Q_norm scale: (34, 256)
K_norm scale: (34, 256)
Gate_proj: (34, 2560, 10240)
Up_proj: (34, 2560, 10240)
Down_proj: (34, 10240, 2560)
Input_norm scale: (34, 2560)
Post_attn_norm scale: (34, 2560)
Pre_ff_norm scale: (34, 2560)
Post_ff_norm scale: (34, 2560)


In [14]:
jax.debug.visualize_array_sharding(m['layers']['input_norm']['scale'])

┌──────────────────────────────────────────────────────────────────────────────┐
│                                                                              │
│                                                                              │
│                                                                              │
│                                                                              │
│                                   GPU 0,1                                    │
│                                                                              │
│                                                                              │
│                                                                              │
│                                                                              │
└──────────────────────────────────────────────────────────────────────────────┘

### Gemma 3 Text:

- 5:1 Interleaved Attention Pattern: Local Layers: 5 out of every 6 layers use a Sliding Window Attention (SWA) with a fixed span of 1024 tokens. Global Layers: Only every 6th layer is a "Global" layer that attends to the full 128k context.

- Frequency-Split RoPE (Rotary Embeddings): Global Layers: Use a base frequency of 1,000,000 (1M). This high frequency allows the model to distinguish between tokens that are very far apart in a 32K sequence. Local Layers: Use a base frequency of 10,000 (10k). Since they only ever look at 512 tokens, they don't need the high-frequency resolution of the global layers.

RMSNorm is a simplified version of LayerNorm. While standard LayerNorm re-centers a vector (subtracts the mean) and then re-scales it (divides by variance), RMSNorm only re-scales.

$$\text{RMS}(x) = \sqrt{\frac{1}{d} \sum_{i=1}^{d} x_i^2 + \epsilon}$$

$$\text{RMSNorm}(x) = \frac{x}{\text{RMS}(x)} \cdot \gamma$$
Where $\gamma$ is a learnable scaling parameter and $\epsilon$ is a tiny constant for numerical stability.

In [15]:
def rms_norm(
        x: jnp.ndarray,        # (..., d)
        scale: jnp.ndarray,    # (d,)
        eps: float = 1e-6
    ) -> jnp.ndarray:
    ms = jnp.mean(jnp.square(x), axis=-1, keepdims=True)
    normed = x * jax.lax.rsqrt(ms + eps)
    return normed * (1.0 + scale)


Rotary Positional Embeddings (RoPE) is the piece of math that allows the model to understand the order of words without needing a fixed map of positions. 

Older models (like GPT-2) just added a fixed vector to the word, which acted like a GPS coordinate ($x + \text{pos}$). RoPE is different. Instead of adding a number, it rotates the vectors in a high-dimensional circle.

The math of RoPE is beautiful because when the model compares two words (the dot product), the result only depends on the angle between them (their relative distance), not their absolute position in the sentence. Most important is Sementic Preservation where rotating a vector doesn't change its length or magnitude, only its direction.

In [16]:
def gemma_rotary_embedding(
        position_ids: jnp.ndarray,     # (seq_len,)
        head_dim: int,
        theta: float = 1000000.0,
        attention_scaling: float = 1.0
    ) -> tuple[jnp.ndarray, jnp.ndarray]:
    inv_freq = 1.0 / (theta ** (jnp.arange(0, head_dim, 2) / head_dim))
    t = jnp.arange(position_ids.max() + 1)
    freqs = jnp.outer(t, inv_freq)
    emb = jnp.concatenate([freqs, freqs], axis=-1)
    cos = (jnp.cos(emb[position_ids]) * attention_scaling).astype(jnp.bfloat16)
    sin = (jnp.sin(emb[position_ids]) * attention_scaling).astype(jnp.bfloat16)
    return cos, sin


In [17]:
def apply_rope(
        x: jnp.ndarray,    # (batch, seq_len, num_heads, head_dim)
        cos: jnp.ndarray,  # (seq_len, head_dim)
        sin: jnp.ndarray   # (seq_len, head_dim)
    ) -> jnp.ndarray:
    cos = cos[None, :, None, :]   # (1, seq_len, 1, head_dim)
    sin = sin[None, :, None, :]   # (1, seq_len, 1, head_dim)
    half = x.shape[-1] // 2
    x1 = x[..., :half]
    x2 = x[..., half:]
    rotate_half_x = jnp.concatenate([-x2, x1], axis=-1)
    return x * cos + rotate_half_x * sin


In JAX, we implement this by creating a precomputed "rotation matrix" and applying it to your $Q$ and $K$ vectors before the attention score calculation.

In [18]:
rotary_embeddings_table = {
    'global': gemma_rotary_embedding(jnp.arange(config.max_seq_len), config.head_dim, theta=1000000.0),
    'local': gemma_rotary_embedding(jnp.arange(config.max_seq_len), config.head_dim, theta=10000.0)
}

In [19]:
def gemma_rotary_embedding_lookup(
        position_ids: jnp.ndarray,                          # (seq_len,)
        embeddings_table: tuple[jnp.ndarray, jnp.ndarray]   # each (max_seq_len, head_dim)
    ) -> tuple[jnp.ndarray, jnp.ndarray]:
    cos_table, sin_table = embeddings_table
    cos = cos_table[position_ids]
    sin = sin_table[position_ids]
    return cos, sin


The padding mask indicates which key positions are real tokens vs padding. During training we attend over the full sequence, so the mask shape is `(batch, 1, 1, seq_len)` — it broadcasts directly against the attention weights `(batch, heads, seq_q, seq_k)` to zero out any column that corresponds to a padding token.


In [20]:
def get_padding_mask(
        input_ids: jnp.ndarray,    # (batch, seq_len)
    ) -> jnp.ndarray:              # (batch, 1, 1, seq_len)
    # 1 for real tokens, 0 for padding — broadcasts against (batch, heads, seq_q, seq_k)
    return (input_ids > 0).astype(jnp.bfloat16)[:, None, None, :]


During training we always process the full sequence at once (no cache, no `start_pos`), so the causal mask is simply a `seq_len × seq_len` lower-triangular matrix. Sliding-window layers further restrict each query to only attend to the nearest `window_size` tokens. The output shape `(1, 1, seq_len, seq_len)` broadcasts over both the batch and head dimensions.


In [21]:
def get_causal_mask(
        seq_len: int,
        is_sliding: bool,
        window_size: int
    ) -> jnp.ndarray:              # (1, 1, seq_len, seq_len)
    q_idx = jnp.arange(seq_len)[:, None]
    k_idx = jnp.arange(seq_len)[None, :]

    sliding_mask = ((q_idx >= k_idx) & (q_idx - window_size < k_idx)).astype(jnp.bfloat16)
    global_mask  = (q_idx >= k_idx).astype(jnp.bfloat16)

    causal_mask = jax.lax.select(is_sliding, sliding_mask, global_mask)
    return causal_mask[None, None, :, :]   # (1, 1, seq_len, seq_len) — broadcasts over batch + heads


In [22]:
def soft_cap(logits, cap_value):
    logits = cap_value * jnp.tanh(logits / cap_value)
    return logits

In [23]:
def gemma_attention_block(
        hidden_states: jnp.ndarray,                             # (batch, seq_len, hidden_size)
        layer: dict[str, jnp.ndarray],
        position_embeddings: tuple[jnp.ndarray, jnp.ndarray],   # each (seq_len, head_dim)
        causal_mask: jnp.ndarray,                               # (1, 1, seq_len, seq_len)
        padding_mask: jnp.ndarray                               # (batch, 1, 1, seq_len)
    ) -> jnp.ndarray:

    batch, seq_len, _ = hidden_states.shape

    q = jnp.einsum('bsd,dh->bsh', hidden_states, layer['q_proj'])
    k = jnp.einsum('bsd,dh->bsh', hidden_states, layer['k_proj'])
    v = jnp.einsum('bsd,dh->bsh', hidden_states, layer['v_proj'])

    q = q.reshape(batch, seq_len, config.num_heads, config.head_dim)
    k = k.reshape(batch, seq_len, config.num_kv_heads, config.head_dim)
    v = v.reshape(batch, seq_len, config.num_kv_heads, config.head_dim)

    q = rms_norm(q, layer['q_norm']['scale'])
    k = rms_norm(k, layer['k_norm']['scale'])

    cos, sin = position_embeddings
    q = apply_rope(q, cos, sin)
    k = apply_rope(k, cos, sin)

    k = jnp.repeat(k, config.num_heads // config.num_kv_heads, axis=2)
    v = jnp.repeat(v, config.num_heads // config.num_kv_heads, axis=2)

    q = q.transpose(0, 2, 1, 3)
    k = k.transpose(0, 2, 1, 3)
    v = v.transpose(0, 2, 1, 3)

    logits = jnp.matmul(q, k.transpose(0, 1, 3, 2)) / jnp.sqrt(config.query_pre_attn_scalar)
    logits = jax.lax.with_sharding_constraint(logits, NamedSharding(mesh, act_att))

    logits = logits + (1.0 - causal_mask) * -1e10
    logits = logits + (1.0 - padding_mask) * -1e10

    logits = soft_cap(logits, 50.0)
    weights = jax.nn.softmax(logits, axis=-1)
    weights = jax.lax.with_sharding_constraint(weights, NamedSharding(mesh, act_att))

    output = jnp.matmul(weights, v)                            # (batch, num_heads, seq_len, head_dim)
    output = output.transpose(0, 2, 1, 3).reshape(batch, seq_len, -1)

    output = jnp.einsum('bsd,dh->bsh', output, layer['o_proj'])
    return output


**Linear Attention** replaces the softmax kernel with a linear feature map $\phi$, reducing complexity from $O(n^2 d)$ to $O(n d^2)$.

Standard attention computes:

$$\text{Attn}(Q,K,V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d}}\right)V$$

Linear attention instead uses:

$$\text{LinAttn}(Q,K,V) = \frac{\phi(Q)\,\bigl(\phi(K)^\top V\bigr)}{\phi(Q)\,\bigl(\phi(K)^\top \mathbf{1}\bigr)}$$

where $\phi(x) = \text{ELU}(x) + 1$ is a non-negative feature map. Computing $(\phi(K)^\top V)$ first gives us the $d \times d$ context matrix independent of sequence length — no attention matrix is ever materialised.


In [24]:
def gemma_linear_attention_block(
        hidden_states: jnp.ndarray,                              # (batch, seq_len, hidden_size)
        layer: dict[str, jnp.ndarray],
        position_embeddings: tuple[jnp.ndarray, jnp.ndarray],    # each (seq_len, head_dim)
        padding_mask: jnp.ndarray                                # (batch, 1, 1, seq_len)
    ) -> jnp.ndarray:                                            # (batch, seq_len, hidden_size)

    batch, seq_len, _ = hidden_states.shape

    q = jnp.einsum('bsd,dh->bsh', hidden_states, layer['q_proj'])
    k = jnp.einsum('bsd,dh->bsh', hidden_states, layer['k_proj'])
    v = jnp.einsum('bsd,dh->bsh', hidden_states, layer['v_proj'])

    q = q.reshape(batch, seq_len, config.num_heads, config.head_dim)
    k = k.reshape(batch, seq_len, config.num_kv_heads, config.head_dim)
    v = v.reshape(batch, seq_len, config.num_kv_heads, config.head_dim)

    q = rms_norm(q, layer['q_norm']['scale'])
    k = rms_norm(k, layer['k_norm']['scale'])

    cos, sin = position_embeddings
    q = apply_rope(q, cos, sin)
    k = apply_rope(k, cos, sin)

    k = jnp.repeat(k, config.num_heads // config.num_kv_heads, axis=2)
    v = jnp.repeat(v, config.num_heads // config.num_kv_heads, axis=2)

    phi_q = jax.nn.elu(q) + 1.0    # (batch, seq_len, num_heads, head_dim)
    phi_k = jax.nn.elu(k) + 1.0    # (batch, seq_len, num_heads, head_dim)

    pad = padding_mask.transpose(0, 3, 1, 2)            # (batch, seq_len, 1, 1)
    phi_k = phi_k * pad                                 # broadcast over heads & head_dim

    kv = jnp.einsum('bshd,bshe->bshde', phi_k, v)       # outer product per token

    S = jnp.cumsum(kv, axis=1)                          # (batch, seq_len, num_heads, head_dim, head_dim)
    z = jnp.cumsum(phi_k, axis=1)                       # (batch, seq_len, num_heads, head_dim)

    numer = jnp.einsum('bshd,bshde->bshe', phi_q, S)
    denom = jnp.einsum('bshd,bshd->bsh', phi_q, z)[..., None]
    denom = jnp.maximum(denom, 1e-6)                

    output = numer / denom                              # (batch, seq_len, num_heads, head_dim)
    output = output.reshape(batch, seq_len, -1)
    output = jnp.einsum('bsd,dh->bsh', output, layer['o_proj'])
    return output


Feedforward network (MLP) implementation. This is a simple two-layer feedforward network with a GELU activation in between. The "gate_proj" and "up_proj" are the two linear transformations, and the output is projected back to the hidden size using "down_proj". The gating mechanism allows for more expressive power in the MLP, as it can modulate the activations based on the input.

In [25]:
def gemma_mlp(
        x: jnp.ndarray,              # (batch, seq_len, hidden_size)
        params: dict[str, jnp.ndarray]
    ) -> jnp.ndarray:
    gate = jnp.einsum('bsd,di->bsi', x, params['gate_proj'])
    up   = jnp.einsum('bsd,di->bsi', x, params['up_proj'])
    activated = jax.nn.gelu(gate, approximate=True) * up
    activated = jax.lax.with_sharding_constraint(activated, NamedSharding(mesh, act_hidden))
    hidden_states = jnp.einsum('bsi,id->bsd', activated, params['down_proj'])
    hidden_states = jax.lax.with_sharding_constraint(hidden_states, NamedSharding(mesh, act_hidden))
    return hidden_states


In [26]:
def gemma_decoder_block(
        hidden_states: jnp.ndarray,                             # (batch, seq_len, hidden_size)
        layer: dict[str, jnp.ndarray],
        position_embeddings: tuple[jnp.ndarray, jnp.ndarray],   # each (seq_len, head_dim)
        causal_mask: jnp.ndarray,                               # (1, 1, seq_len, seq_len)
        padding_mask: jnp.ndarray                               # (batch, 1, 1, seq_len)
    ) -> jnp.ndarray:

    residual = hidden_states
    hidden_states = rms_norm(hidden_states, layer['input_norm']['scale'])
    hidden_states = gemma_attention_block(hidden_states, layer['attn'], position_embeddings, causal_mask, padding_mask)
    hidden_states = rms_norm(hidden_states, layer['post_attn_norm']['scale']) + residual

    residual = hidden_states
    hidden_states = rms_norm(hidden_states, layer['pre_ff_norm']['scale'])
    hidden_states = gemma_mlp(hidden_states, layer['mlp'])
    hidden_states = rms_norm(hidden_states, layer['post_ff_norm']['scale']) + residual

    return hidden_states


In [ ]:
def gemma_forward(
        inputs_ids: jnp.ndarray,                     # (batch, seq_len)
        m: dict[str, jnp.ndarray],
        config: Gemma3Config,
        rotary_embeddings_table: dict[str, jnp.ndarray],
    ) -> jnp.ndarray:                                # (batch, seq_len, vocab_size)

    hidden_states = m['embed'][inputs_ids] * jnp.sqrt(config.hidden_size)  # (batch, seq_len, hidden_size)
    hidden_states = jax.lax.with_sharding_constraint(hidden_states, NamedSharding(mesh, act_hidden))

    seq_len = hidden_states.shape[1]
    position_ids = jnp.arange(seq_len)

    pos_emb_local  = gemma_rotary_embedding_lookup(position_ids, rotary_embeddings_table['local'])
    pos_emb_global = gemma_rotary_embedding_lookup(position_ids, rotary_embeddings_table['global'])

    padding_mask = get_padding_mask(inputs_ids)        # (batch, 1, 1, seq_len)

    # is_sliding is a config property, not a learned weight — compute from config
    is_sliding_flags = jnp.array([i not in config.full_attention_layers for i in range(config.num_layers)])

    def scan_fn(hidden_states, xs):
        layer, is_sliding = xs
        causal_mask = get_causal_mask(seq_len, is_sliding, config.sliding_window)

        cos = jax.lax.select(is_sliding, pos_emb_local[0], pos_emb_global[0])
        sin = jax.lax.select(is_sliding, pos_emb_local[1], pos_emb_global[1])

        hidden_states = gemma_decoder_block(
            hidden_states,
            layer,
            (cos, sin),
            causal_mask,
            padding_mask
        )
        hidden_states = jax.lax.with_sharding_constraint(hidden_states, NamedSharding(mesh, act_hidden))
        return hidden_states, None

    hidden_states, _ = jax.lax.scan(scan_fn, hidden_states, (m['layers'], is_sliding_flags))

    hidden_states = rms_norm(hidden_states, m['final_norm'])

    logits = jnp.einsum('bsd,vd->bsv', hidden_states, m['embed'])  # (batch, seq_len, vocab_size)    
    logits = jax.lax.with_sharding_constraint(
        logits, NamedSharding(mesh, jax.P("fsdp", None, None))
    )
    return logits

This function takes the raw logits output from the model and samples a token index based on the specified temperature. A lower temperature will make the sampling more deterministic (favoring higher probability tokens), while a higher temperature will make it more random. 

The function uses JAX's random categorical sampling to select a token index according to the probabilities derived from the logits.

In [29]:
@partial(jax.jit, static_argnums=(1,))
def sample_token(
        logits: jnp.ndarray,    # (vocab_size,)
        temperature: float = 1.0,
        key: jax.random.PRNGKey = None
    ) -> jnp.ndarray:
    if key is None:
        key = jax.random.PRNGKey(0)
    scaled_logits = logits / temperature
    probs = jax.nn.softmax(scaled_logits)
    token = jax.random.categorical(key, jnp.log(probs))
    return token


The prepare_input function below takes a raw text prompt, tokenizes it, and prepares it for input into the model. It converts the prompt into token IDs, pads them to a fixed length (MAX_PROMPT_LEN), and returns both the padded token IDs and the actual length of the original token sequence. This is important because the model needs fixed-size inputs, but we also want to keep track of how much of that input is real data versus padding.

In [30]:
MAX_PROMPT_LEN = config.max_seq_len

In [30]:
def prepare_input(
        prompts: list[str]
    ) -> tuple[jnp.ndarray, list[int]]:  # (batch, MAX_PROMPT_LEN), [actual_len, ...]
    enc = tokenizer(
        prompts,
        return_tensors="np",
        padding="max_length",
        truncation=True,
        max_length=MAX_PROMPT_LEN
    )
    input_ids = jnp.array(enc['input_ids'])                    # (batch, MAX_PROMPT_LEN)
    actual_lens = enc['attention_mask'].sum(axis=-1).tolist()  # [actual_len per prompt]
    return input_ids, actual_lens


In [32]:
prompts = ["Tell me a story?", "Hey, how are you?", "What is happening?", "Good morning"]
input_ids, actual_lens = prepare_input(prompts)

In [33]:
input_ids.shape

(4, 512)

In [34]:
jitted_gemma_forward = jax.jit(
    gemma_forward, 
    static_argnames=['config']
)

In [36]:
out = jitted_gemma_forward(input_ids, m, config, rotary_embeddings_table)
out.block_until_ready()
del out

In [ ]:
# Single forward pass — returns (batch, seq_len, vocab_size)
logits = gemma_forward(input_ids, m, config, rotary_embeddings_table)
print(f"logits shape: {logits.shape}")   # (1, seq_len, vocab_size)

In [35]:
import jax.profiler

# 1. Warm up (JIT compile)
# Ensure your hidden_states and params are already sharded
_ = jitted_gemma_forward(input_ids, m, config, rotary_embeddings_table) 
jax.block_until_ready(_)

# 2. Start the Trace
# This creates a folder with the raw profile data
with jax.profiler.trace("gemma_trace"):
    # Run 2-3 iterations to see the 'steady state'
    for _ in range(3):
        out = jitted_gemma_forward(input_ids, m, config, rotary_embeddings_table)
        jax.block_until_ready(out) # Force synchronization for accurate timing
        del out

### Dataset

In [31]:
from datasets import load_dataset

CSV_PATH = "medtext_2.csv"

ds = load_dataset("BI55/MedText", split="train")
print(f"Rows: {len(ds)}")
print(f"Columns: {ds.column_names}")
print(ds[0])

Rows: 1412
Columns: ['Prompt', 'Completion']
{'Prompt': 'A 50-year-old male presents with a history of recurrent kidney stones and osteopenia. He has been taking high-dose vitamin D supplements due to a previous diagnosis of vitamin D deficiency. Laboratory results reveal hypercalcemia and hypercalciuria. What is the likely diagnosis, and what is the treatment?', 'Completion': "This patient's history of recurrent kidney stones, osteopenia, and high-dose vitamin D supplementation, along with laboratory findings of hypercalcemia and hypercalciuria, suggest the possibility of vitamin D toxicity. Excessive intake of vitamin D can cause increased absorption of calcium from the gut, leading to hypercalcemia and hypercalciuria, which can result in kidney stones and bone loss. Treatment would involve stopping the vitamin D supplementation and potentially providing intravenous fluids and loop diuretics to promote the excretion of calcium."}


Each row has a `Prompt` (patient symptoms) and a `Completion` (diagnosis + treatment). We format them as a single string so the model learns to generate the completion given the presentation. We use the Gemma chat template format wrapping the text in `<bos>` and `<eos>` tokens so the model knows where sequences start and end.

For training a causal LM, `labels` are simply `input_ids` shifted left by one — each token predicts the next. We mask out padding positions in the labels with `-100` so they are ignored by the loss.


In [32]:
MAX_SEQ_LEN = 512

BOS = tokenizer.bos_token  # <bos>
EOS = tokenizer.eos_token  # <eos>

tokenizer.padding_side = "right"

def format_example(row):
    text = f"{BOS}Presentation: {row['Prompt']}\nCompletion: {row['Completion']}{EOS}"
    return {"text": text}

def tokenize(row):
    enc = tokenizer(
        row["text"],
        max_length=MAX_SEQ_LEN,
        truncation=True,
        padding="max_length",
    )
    input_ids = enc["input_ids"]                # list[int], len = MAX_SEQ_LEN
    attention_mask = enc["attention_mask"]      # 1 = real, 0 = pad

    labels = input_ids[1:] + [-100]
    labels = [t if attention_mask[i] == 1 else -100 for i, t in enumerate(labels)]

    return {"input_ids": input_ids, "labels": labels, "attention_mask": attention_mask}

formatted = ds.map(format_example, remove_columns=ds.column_names)
tokenized = formatted.map(tokenize, remove_columns=["text"])
tokenized.set_format("numpy")

print(f"Tokenized dataset: {len(tokenized)} examples, seq_len = {MAX_SEQ_LEN}")


Tokenized dataset: 1412 examples, seq_len = 512


The dataloader shuffles the tokenized examples and yields fixed-size batches of JAX arrays. It drops the last incomplete batch so every batch is exactly `(batch_size, MAX_SEQ_LEN)`. `input_ids` feeds the model; `labels` are used to compute the cross-entropy loss.


In [33]:
def data_loader(
    dataset, 
    batch_size: int, 
    shuffle: bool = True, 
    seed: int = 0
):
    indices = np.arange(len(dataset))
    if shuffle:
        rng = np.random.default_rng(seed)
        rng.shuffle(indices)

    for start in range(0, len(indices) - batch_size + 1, batch_size):
        batch_idx = indices[start : start + batch_size]
        batch = dataset[batch_idx]
        yield (
            jnp.array(batch["input_ids"],  dtype=jnp.int32),   # (batch, seq_len)
            jnp.array(batch["labels"],     dtype=jnp.int32),   # (batch, seq_len)
        )

BATCH_SIZE = 16

# Sanity check — peek at first batch
for input_ids_batch, labels_batch in data_loader(tokenized, BATCH_SIZE):
    print(f"input_ids : {input_ids_batch.shape}")
    print(f"labels    : {labels_batch.shape}")
    print(f"sample ids  : {input_ids_batch[1, :8]}")
    print(f"sample labels: {labels_batch[1, :8]}")
    break


input_ids : (16, 512)
labels    : (16, 512)
sample ids  : [     2      2  81167 236787    562 236743 236800 236771]
sample labels: [     2  81167 236787    562 236743 236800 236771 236772]


### Training

#### Gradient Checkpointing

By default JAX stores every intermediate activation from the forward pass so it can use them during the backward pass. For an 18-layer model this balloons memory quadratically. `jax.remat` (also called `jax.checkpoint`) fixes this: it discards the activations after the forward pass and **recomputes** them on-the-fly during backprop. This trades a small amount of extra FLOPs for a large memory saving, and is essentially free on modern GPUs where compute is cheaper than memory bandwidth.


In [ ]:
gemma_decoder_block = jax.checkpoint(
    gemma_decoder_block,
    policy=jax.checkpoint_policies.nothing_saveable
)

#### Loss Function

We use **sparse softmax cross-entropy** — for each position, the model predicts a probability distribution over the full vocabulary and we penalise it for not assigning high probability to the correct next token.

$$\mathcal{L} = -\frac{1}{N} \sum_{i} \mathbf{1}[y_i \neq -100] \cdot \log p(y_i \mid x_{<i})$$

Where $N$ is the number of non-padding tokens. The `-100` mask means padding positions contribute nothing to the loss — the model is only trained on real tokens.


In [35]:
def cross_entropy_loss(
        logits: jnp.ndarray,   # (batch, seq_len, vocab_size)
        labels: jnp.ndarray    # (batch, seq_len)  — -100 at padding positions
    ) -> jnp.ndarray:          # scalar
    logits = logits.astype(jnp.float32)
    valid = (labels != -100)                                                       # (batch, seq_len) bool mask
    safe_labels = jnp.where(valid, labels, 0)                                      # replace -100 so indexing is valid
    loss = optax.softmax_cross_entropy_with_integer_labels(logits, safe_labels)    # (batch, seq_len)
    return jnp.sum(loss * valid) / jnp.sum(valid)                                  # mean over real tokens only


#### Optimizer

We use **AdamW** — Adam with decoupled weight decay. The key hyperparameters are:

- **Learning rate** `1e-4` — a safe starting point for fine-tuning; lower than pre-training because the weights are already good
- **Weight decay** `0.01` — regularises all parameters *except* norms and biases (which should not be decayed) via `optax.masked`
- **Gradient clipping** `max_norm=1.0` — prevents exploding gradients by rescaling the gradient if its global norm exceeds 1

AdamW is chained with gradient clipping using `optax.chain` so both transformations are applied in a single optimizer step.


In [ ]:
LEARNING_RATE = 1e-4
WEIGHT_DECAY  = 0.01    # AdamW
MAX_GRAD_NORM = 5.0

# Don't apply weight decay to norm scales or biases
def is_not_norm_or_bias(path):
    path_str = "/".join(str(k) for k in path)
    return "norm" not in path_str and "bias" not in path_str

optimizer = optax.chain(
    optax.clip_by_global_norm(MAX_GRAD_NORM),
    optax.adafactor(learning_rate=LEARNING_RATE),
)

opt_state = optimizer.init(m)

print("Optimizer initialised.")
print(f"  lr={LEARNING_RATE}  weight_decay={WEIGHT_DECAY}  grad_clip={MAX_GRAD_NORM}")

Optimizer initialised.
  lr=5e-05  weight_decay=0.01  grad_clip=5.0


#### Train Step

`train_step` is the innermost function that runs once per batch. It is fully `jax.jit` compiled so XLA traces it once and then executes it as a single fused GPU kernel with no Python overhead.

Inside we use `jax.value_and_grad` which computes both the loss scalar **and** the gradients with respect to all model parameters in a single forward+backward pass — this is equivalent to `loss.backward()` in PyTorch but entirely functional (no mutation of any state).

The optimizer then converts gradients into parameter updates and we apply them to get the new parameters. All three outputs — `new_params`, `new_opt_state`, `loss` — are pure return values; nothing is mutated.


#### Stochastic Rounding

When weights are stored as `bfloat16`, the optimizer update is computed in `float32` then cast back to `bfloat16`. Standard casting always truncates (rounds toward zero), introducing a **systematic downward bias** that accumulates step after step.

**Stochastic rounding** fixes this by rounding up or down with probability proportional to proximity:

$$\text{SR}(x) = \begin{cases} \lceil x \rceil_{bf16} & \text{with prob } \dfrac{x - \lfloor x \rfloor_{bf16}}{\text{ulp}(x)} \\[6pt] \lfloor x \rfloor_{bf16} & \text{otherwise} \end{cases}$$

This makes the rounding **unbiased in expectation** — small gradient updates that would otherwise always round to zero have a chance of being applied.

**Implementation trick:** `bfloat16` keeps the top 16 bits of a `float32` and drops the bottom 16 mantissa bits. So we add a uniform random integer in $[0, 2^{16})$ directly to the raw `int32` bit pattern of `x` before casting — the random carry into the upper 16 bits is exactly the stochastic rounding probability.


In [ ]:
def stochastic_round_bf16(
    x: jnp.ndarray, 
    key: jax.random.key
):
    low_bf16  = x.astype(jnp.bfloat16)              # floor in bf16 space
    low_f32   = low_bf16.astype(jnp.float32)

    # next representable bf16 above low: add 1 ULP to the raw uint16 bits
    high_f32  = jax.lax.bitcast_convert_type(
                jax.lax.bitcast_convert_type(low_bf16, jnp.uint16) + jnp.uint16(1),
                jnp.bfloat16
            ).astype(jnp.float32)

    gap  = high_f32 - low_f32                                   # bf16 ULP at this magnitude
    prob = jnp.where(gap > 0, (x - low_f32) / gap, 0.0)         # fractional position in [low, high]
    roll = jax.random.uniform(key, shape=x.shape)
    return jnp.where(roll < prob, high_f32, low_f32).astype(jnp.bfloat16)

def apply_stochastic_rounding(params, key):
    """Applied in Python, outside JIT, so no tracing overhead."""
    leaves, treedef = jax.tree_util.tree_flatten(params)
    leaf_keys = jax.random.split(key, len(leaves))
    rounded = [
        stochastic_round_bf16(leaf, lk) if leaf.dtype == jnp.bfloat16 else leaf
        for leaf, lk in zip(leaves, leaf_keys)
    ]
    return treedef.unflatten(rounded), jax.random.fold_in(key, 1)

In [39]:
@jax.jit
def train_step(
        params: dict,
        opt_state,
        input_ids: jnp.ndarray,   # (batch, seq_len)
        labels: jnp.ndarray,      # (batch, seq_len)
        key: jax.random.PRNGKey
    ):
    def loss_fn(params):
        logits = gemma_forward(input_ids, params, config, rotary_embeddings_table)
        return cross_entropy_loss(logits, labels)

    loss, grads = jax.value_and_grad(loss_fn)(params)
    grad_norm = optax.global_norm(grads)

    updates, new_opt_state = optimizer.update(grads, opt_state, params)
    new_params = optax.apply_updates(params, updates)

    return new_params, new_opt_state, loss, grad_norm


#### Training Loop

The outer loop iterates over epochs, the inner loop iterates over batches from the dataloader. The first call to `train_step` will trigger XLA compilation (takes ~30–60s); every subsequent call runs the pre-compiled kernel and is very fast.

Key things logged per step:
- **loss** — the raw cross-entropy value; should decrease over training
- **step time** — time per batch; useful to spot slowdowns
- **tokens/sec** — throughput metric; `batch_size × seq_len / step_time`

The seed passed to `data_loader` changes each epoch so the batch order is reshuffled every pass through the data.


In [40]:
import pynvml

# ── Hardware constants ─────────────────────────────────────────────
NUM_GPUS     = 2
PEAK_FLOPS_PER_GPU = 71e12          # RTX 3090 bfloat16
PEAK_FLOPS   = PEAK_FLOPS_PER_GPU * NUM_GPUS   # total system FLOPS

# ── Parameter count ────────────────────────────────────────────────
# jax.tree_util.tree_leaves returns sharded arrays — .size on a sharded
# array returns the GLOBAL size (all shards combined), not the local shard.
# So sum(x.size) is already correct for global param count.
num_params = sum(x.size for x in jax.tree_util.tree_leaves(m))

# 6 * params * tokens is the standard FLOPs estimate for a transformer
# (forward + backward ≈ 3x forward, forward ≈ 2 * params * tokens)
flops_per_step = 6 * num_params * (BATCH_SIZE * MAX_SEQ_LEN)

print(f"Model parameters:  {num_params:,}")
print(f"FLOPs per step:    {flops_per_step:.2e}")
print(f"Peak FLOPs (total): {PEAK_FLOPS:.2e}  ({NUM_GPUS}x RTX 3090)")

# ── pynvml handles for all GPUs ────────────────────────────────────
pynvml.nvmlInit()
gpu_handles = [pynvml.nvmlDeviceGetHandleByIndex(i) for i in range(NUM_GPUS)]

def get_total_power_watts():
    """Sum power across all GPUs in watts."""
    return sum(pynvml.nvmlDeviceGetPowerUsage(h) / 1000 for h in gpu_handles)

def get_gpu_stats():
    """Per-GPU memory and power for logging."""
    stats = []
    for i, h in enumerate(gpu_handles):
        mem   = pynvml.nvmlDeviceGetMemoryInfo(h)
        power = pynvml.nvmlDeviceGetPowerUsage(h) / 1000
        stats.append({
            "gpu": i,
            "mem_used_gb": mem.used / 1024**3,
            "mem_total_gb": mem.total / 1024**3,
            "power_w": power,
        })
    return stats

Model parameters:  3,880,263,168
FLOPs per step:    1.91e+14
Peak FLOPs (total): 1.42e+14  (2x RTX 3090)


In [ ]:
import time

NUM_EPOCHS  = 3
LOG_EVERY   = 10   # print loss every N steps

global_step = 0
losses = []
key = jax.random.PRNGKey(42)   # stochastic rounding PRNG

for epoch in range(NUM_EPOCHS):
    epoch_start = time.time()
    epoch_losses = []

    for input_ids_batch, labels_batch in data_loader(tokenized, BATCH_SIZE, seed=epoch):
        
        input_ids_batch = jax.device_put(input_ids_batch, NamedSharding(mesh, P("fsdp", None)))
        labels_batch = jax.device_put(labels_batch, NamedSharding(mesh, P("fsdp", None)))
        
        step_start = time.time()

        m, opt_state, loss, grad_norm = train_step(m, opt_state, input_ids_batch, labels_batch, key)
        m, key = apply_stochastic_rounding(m, key)
        
        loss, grad_norm = jax.block_until_ready((loss, grad_norm))
        loss_val = float(loss)
        grad_norm_val = float(grad_norm)
        del loss, grad_norm    

        epoch_losses.append(loss_val)

        step_time       = time.time() - step_start
        tokens_per_sec  = (BATCH_SIZE * MAX_SEQ_LEN) / step_time
        perplexity      = float(jnp.exp(loss_val))

        # ── power across BOTH GPUs ──
        total_power_w   = get_total_power_watts()
        tokens_per_watt = tokens_per_sec / total_power_w

        # ── MFU against TOTAL system FLOPS ──
        mfu = (flops_per_step / step_time) / PEAK_FLOPS

        if global_step % LOG_EVERY == 0:
            gpu_stats = get_gpu_stats()
            mem_str = "  ".join(
                f"GPU{s['gpu']}: {s['mem_used_gb']:.1f}/{s['mem_total_gb']:.0f}GB {s['power_w']:.0f}W"
                for s in gpu_stats
            )
            print(
                f"epoch {epoch+1}/{NUM_EPOCHS}  "
                f"step {global_step:4d}  "
                f"loss {loss_val:.4f}  "
                f"ppl {perplexity:.2f}  "
                f"grad_norm {grad_norm_val:.2f}  "
                f"{tokens_per_sec:,.0f} tok/sec  "
                f"{tokens_per_watt:.1f} tok/watt  "
                f"mfu {mfu:.3f}  "
                f"[{mem_str}]"
            )

        global_step += 1

    epoch_time = time.time() - epoch_start
    avg_loss   = sum(epoch_losses) / len(epoch_losses)
    print(f"\n── epoch {epoch+1} done in {epoch_time:.1f}s  avg_loss={avg_loss:.4f}\n")

print("Training complete.")


E0326 04:31:01.047719   34549 xtile_compiler.cc:399] Fusion: gemm_fusion_dot.82 = bf16[4096,2560]{1,0} fusion(bitcast.1383, bitcast.94, bitcast.97, bitcast.90), kind=kCustom, calls=gemm_fusion_dot.82_computation.clone, backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"fusion_backend_config":{"kind":"__triton_nested_gemm_fusion","block_level_fusion_config":{"num_warps":"8","output_tiles":[{"sizes":["128","256"]}],"num_ctas":1,"num_stages":4,"is_tma_allowed":false,"is_warp_specialization_allowed":false}},"force_earliest_schedule":false,"reification_cost":[],"device_type":"DEVICE_TYPE_INVALID"}
E0326 04:31:01.047874   34549 xtile_compiler.cc:401] Computation: gemm_fusion_dot.82_computation.clone {
  parameter_1.25 = bf16[4096,1024]{1,0} parameter(1)
  parameter_2.2 = bf16[4096,2048]{1,0} parameter(2)
  parameter_3.2 = bf16[4096,1024]{1,0} parameter(3)
  concatenate.42 = bf16[4096,4096]{1,0} concatenate(parameter_1.25, parameter_2.2, parameter_3.2), dimensions={1}
  p

epoch 1/3  step    0  loss 2.7009  ppl 14.89  grad_norm 44.00  146 tok/sec  0.2 tok/watt  mfu 0.024  [GPU0: 16.9/24GB 310W  GPU1: 16.9/24GB 313W]


W0326 04:31:53.919618   33759 hlo_rematerialization.cc:3233] Can't reduce memory use below 13.53GiB (14527470449 bytes) by rematerialization; only reduced to 15.58GiB (16727760270 bytes), down from 15.60GiB (16748772766 bytes) originally


epoch 1/3  step   10  loss 2.4145  ppl 11.18  grad_norm 34.25  2,073 tok/sec  3.3 tok/watt  mfu 0.340  [GPU0: 16.9/24GB 316W  GPU1: 16.9/24GB 321W]
epoch 1/3  step   20  loss 2.3396  ppl 10.38  grad_norm 31.00  2,069 tok/sec  3.2 tok/watt  mfu 0.339  [GPU0: 16.9/24GB 312W  GPU1: 16.9/24GB 325W]
epoch 1/3  step   30  loss 2.0780  ppl 7.99  grad_norm 27.12  2,071 tok/sec  3.3 tok/watt  mfu 0.339  [GPU0: 16.9/24GB 313W  GPU1: 16.9/24GB 319W]
epoch 1/3  step   40  loss 1.9917  ppl 7.33  grad_norm 53.75  2,069 tok/sec  3.2 tok/watt  mfu 0.339  [GPU0: 16.9/24GB 312W  GPU1: 16.9/24GB 326W]
epoch 1/3  step   50  loss 1.9476  ppl 7.01  grad_norm 32.00  2,069 tok/sec  3.3 tok/watt  mfu 0.339  [GPU0: 16.9/24GB 311W  GPU1: 16.9/24GB 325W]
epoch 1/3  step   60  loss 1.8606  ppl 6.43  grad_norm 39.75  2,068 tok/sec  3.3 tok/watt  mfu 0.339  [GPU0: 16.9/24GB 307W  GPU1: 16.9/24GB 321W]
